In [212]:
import duckdb
import pandas as pd

con = duckdb.connect()

# Foul and Turnover Study  
**Currently the 7s/11s shot quality buckets do not accurately reflect (enough) the value of what occurs. In additon to that, there is inquiry into the added value of "more" through turnovers (since rebounds are already tracked).**

The file below contains additional data tracked after the season ended with a 'Type' column for both Home and Away.  
In the 'Type' column(s):
- Fouls are tracked via:
    - F2: Foul on 2PA (not an And-1)
    - F2A: Foul on an And-1 2PA
    - F3: Foul on 3PA (not an And-1)
    - F3A: Foul on an And-1 3PA
    - OAO: One-and-One Bonus
    - DB: Double Bonus
    - T1: One-Shot Technical
    - T2: Two-Shot Technical  
- Turnovers are tracked via:
    - LF: Live-Ball Turnovers in the Full Court
    - LH: Live-Ball Turnovers in the Half Court
    - DF: Dead-Ball Turnovers in the Full Court
    - DH: Dead-Ball Turnovers in the Half Court

In [213]:
file = "Fouls_and_TOs.csv"

In [214]:
data_q = f"""
    SELECT Date, Home_Team, Home_Type, Home_Points, Away_Team, Away_Type, Away_Points FROM {file}
    WHERE Date IS NOT NULL
"""
data = con.query(data_q)

In [215]:
foul_h_query = f"""
    SELECT Home_Team, Home_Type, sum(Home_Points) as H_Total_Points, count(Home_Points) as H_Total_Shots FROM data
    WHERE Home_Type in ('T1', 'T2', 'F2', 'F2A', 'F3', 'F3A', 'OAO', 'DB')
    GROUP BY Home_Team, Home_Type
"""

#con.sql(foul_h_query).show()

In [216]:
foul_a_query = f"""
    SELECT Away_Team, Away_Type, sum(Away_Points) as 'A_Total_Points', count(Away_Points) as 'A_Total_Shots' FROM data
    WHERE Away_Type in ('T1', 'T2', 'F2', 'F2A', 'F3', 'F3A', 'OAO', 'DB')
    GROUP BY Away_Team, Away_Type
"""

#con.sql(foul_a_query)

In [217]:
team_totals = f"""
    WITH Home as ({foul_h_query}),
    Away as ({foul_a_query})
    SELECT COALESCE(h.Home_Team, a.Away_Team) as Team, 
        COALESCE (h.Home_Type, a.Away_Type) as Type, 
        IFNULL(h.H_Total_Points, 0) Home_Points, 
        IFNULL(H_Total_Shots, 0) Home_Shots, 
        IFNULL(a.A_Total_Points, 0) Away_Points, 
        IFNULL(a.A_Total_Shots, 0) Away_Shots,
        (Home_Points+Away_Points) Total_Points,
        (Home_Shots+Away_Shots) Total_Shots
    FROM Home h
    FULL JOIN Away a
    ON h.Home_Team = a.Away_Team
    AND h.Home_Type = a.Away_Type
"""

#con.sql(team_totals)

In [218]:
import numpy as np
from scipy import stats as sps

# Register a UDF DuckDB can call directly in SQL
def t_pvalue(t_stat: float, df: float) -> float:
    if df is None or df <= 0 or t_stat is None:
        return None
    return float(2 * sps.t.sf(abs(t_stat), df))  # two-tailed

con.create_function('t_pvalue', t_pvalue, ['DOUBLE', 'DOUBLE'], 'DOUBLE')

## This code outputs the PPS, equivalent System Score, and the "Mathematical" Expectation for each Foul Type.

In [ ]:
shots_q = f"""
    SELECT Home_Team as Team, Home_Type as Type, Home_Points as Points FROM data
    WHERE Home_Type in ('T1','T2','F2','F2A','F3','F3A','OAO','DB')
    UNION ALL
    SELECT Away_Team as Team, Away_Type as Type, Away_Points as Points FROM data
    WHERE Away_Type in ('T1','T2','F2','F2A','F3','F3A','OAO','DB')
"""

stats_q = f"""
    WITH shots AS ({shots_q}),
    Math AS (
        SELECT * FROM (
            VALUES
                ('T1', 3), ('T2', 6), ('F2', 6), ('F2A', 11), ('F3', 9), ('F3A', 15), ('OAO', 5), ('DB', 6)
        ) AS t(Type, Math_System_Score)
    ),
    Agg AS (
        SELECT s.Type,
               COUNT(*) AS Shots,
               AVG(Points) AS PPS,
               STDDEV_SAMP(Points) AS SD,
               m.Math_System_Score / 4.0 AS Expected_PPS
        FROM shots s
        JOIN Math m ON s.Type = m.Type
        GROUP BY s.Type, m.Math_System_Score
    ),
    T AS (
        SELECT *, (PPS - Expected_PPS) / (SD / SQRT(Shots)) t_stat, (Shots - 1) AS df
        FROM Agg
    )
    SELECT
        Type,
        Shots,
        printf('%.2f', PPS)          AS PPS,
        printf('%.2f', Expected_PPS) AS Expected_PPS,
        printf('%.2f', SD) AS SD,
        printf('%.2f', t_stat)       AS t_stat,
        ROUND(t_pvalue(t_stat, df), 4) AS p_value,
        t_pvalue(t_stat, df) < 0.05/(SELECT COUNT(*) FROM Agg) AS significance
    FROM T
    ORDER BY CASE Type
        WHEN 'F2'  THEN 1
        WHEN 'F2A' THEN 2
        WHEN 'F3'  THEN 3
        WHEN 'F3A' THEN 4
        WHEN 'OAO' THEN 5
        WHEN 'DB'  THEN 6
        WHEN 'T1'  THEN 7
        WHEN 'T2'  THEN 8
    END
"""

# Persist as a view so it acts like a queryable table/DB object
con.sql(stats_q)

┌─────────┬───────┬─────────┬──────────────┬─────────┬─────────┬─────────┬──────────────┐
│  Type   │ Shots │   PPS   │ Expected_PPS │   SD    │ t_stat  │ p_value │ significance │
│ varchar │ int64 │ varchar │   varchar    │ varchar │ varchar │ double  │   boolean    │
├─────────┼───────┼─────────┼──────────────┼─────────┼─────────┼─────────┼──────────────┤
│ F2      │  1847 │ 1.46    │ 1.50         │ 0.64    │ -2.49   │  0.0129 │ false        │
│ F2A     │   456 │ 2.71    │ 2.75         │ 0.45    │ -1.86   │   0.064 │ false        │
│ F3      │    77 │ 2.27    │ 2.25         │ 0.84    │ 0.24    │  0.8124 │ false        │
│ F3A     │    23 │ 3.91    │ 3.75         │ 0.29    │ 2.71    │  0.0127 │ false        │
│ OAO     │   412 │ 1.26    │ 1.25         │ 0.87    │ 0.34    │  0.7355 │ false        │
│ DB      │   326 │ 1.44    │ 1.50         │ 0.63    │ -1.85   │  0.0649 │ false        │
│ T1      │     3 │ 0.67    │ 0.75         │ 0.58    │ -0.25   │  0.8259 │ false        │
│ T2      

Looking at the Foul Study via Type Breakdown, there are some telling findings.
1. There is an alternative way to assess 7s/11s. Upon initial glance, the 8 different foul buckets have varying Expected PPS and therefore do not necessarily belong to a strict 7 or 11 bucket.
2. Using bonferroni p-values (0.05/8 groups) for a significance level, the actual values found

In [220]:
# Reuses team_totals exactly as already defined — no restart
stats_q = f"""
    WITH TeamTotals AS ({team_totals}),
    TeamPPS AS (
        SELECT Team, Type, Total_Points, Total_Shots,
               Total_Points * 1.0 / Total_Shots AS Team_PPS
        FROM TeamTotals
        WHERE Total_Shots > 0
    ),
    Math AS (
        SELECT * FROM (
            VALUES
                ('T1', 3), ('T2', 6), ('F2', 6), ('F2A', 11), ('F3', 9), ('F3A', 15), ('OAO', 5), ('DB', 6)
        ) AS t(Type, Math_System_Score)
    ),
    Agg AS (
        SELECT tp.Type,
               SUM(tp.Total_Shots)                                AS Shots,
               SUM(tp.Total_Points) * 1.0 / SUM(tp.Total_Shots)   AS PPS,
               AVG(tp.Team_PPS)                                    AS Mean_Team_PPS,
               STDDEV_SAMP(tp.Team_PPS)                            AS SD,
               COUNT(*)                                            AS N_Teams,
               m.Math_System_Score,
               m.Math_System_Score / 4.0                           AS Expected_PPS
        FROM TeamPPS tp
        JOIN Math m ON tp.Type = m.Type
        GROUP BY tp.Type, m.Math_System_Score
    ),
    T AS (
        SELECT *,
               (Mean_Team_PPS - Expected_PPS) / (SD / SQRT(N_Teams)) AS t_stat,
               (N_Teams - 1) AS df
        FROM Agg
    )
    SELECT
        Type,
        Shots,
        ROUND(PPS, 3)              AS PPS,
        ROUND(Expected_PPS, 3)     AS Expected_PPS,
        ROUND(4 * PPS)             AS System_Score,
        Math_System_Score,
        ROUND(SD, 3)               AS SD,
        N_Teams,
        df,
        ROUND(t_stat, 3)           AS t_stat,
        ROUND(t_pvalue(t_stat, df), 4) AS p_value,
        (t_pvalue(t_stat, df) < 0.05 / (SELECT COUNT(*) FROM Agg)) AS significance
    FROM T
"""

con.sql(f"CREATE OR REPLACE VIEW stats AS {stats_q}")
con.sql("SELECT * FROM stats ORDER BY p_value")

┌─────────┬────────┬────────┬──────────────┬──────────────┬───┬─────────┬───────┬────────┬─────────┬──────────────┐
│  Type   │ Shots  │  PPS   │ Expected_PPS │ System_Score │ … │ N_Teams │  df   │ t_stat │ p_value │ significance │
│ varchar │ int128 │ double │    double    │    double    │   │  int64  │ int64 │ double │ double  │   boolean    │
├─────────┼────────┼────────┼──────────────┼──────────────┼───┼─────────┼───────┼────────┼─────────┼──────────────┤
│ F3A     │     23 │  3.913 │         3.75 │         16.0 │ … │      10 │     9 │   3.28 │  0.0095 │ false        │
│ F2A     │    456 │  2.711 │         2.75 │         11.0 │ … │      12 │    11 │ -2.217 │  0.0486 │ false        │
│ F2      │   1847 │  1.463 │          1.5 │          6.0 │ … │      12 │    11 │ -2.155 │  0.0541 │ false        │
│ DB      │    326 │  1.436 │          1.5 │          6.0 │ … │      12 │    11 │ -1.322 │  0.2131 │ false        │
│ T2      │     72 │  1.528 │          1.5 │          6.0 │ … │      12 

As seen in the table above, all 8 categories of Foul Type's System Score reflects what the Mathematical System Score should be. This indicates a new way to track foul occurrences in games!

In [221]:
shooting = f"""
    SELECT 
        CASE WHEN Type IN ('F2', 'F2A') THEN 'F_on_2PA'
             WHEN Type IN ('F3', 'F3A') THEN 'F_on_3PA'
        END as Shooting_Type,
        sum(Points) as Points,
        sum(Shots) as Shots,
        ROUND(sum(Points)/sum(Shots), 2) as PPS,
        ROUND(4*ROUND(sum(Points)/sum(Shots), 2)) as System_Score
    FROM ({totals})
    WHERE Type = 'F2' or Type = 'F2A' or Type = 'F3' or Type = 'F3A'
    GROUP BY Shooting_Type
"""

con.sql(shooting)

┌───────────────┬────────┬────────┬────────┬──────────────┐
│ Shooting_Type │ Points │ Shots  │  PPS   │ System_Score │
│    varchar    │ int128 │ int128 │ double │    double    │
├───────────────┼────────┼────────┼────────┼──────────────┤
│ F_on_2PA      │   3938 │   2303 │   1.71 │          7.0 │
│ F_on_3PA      │    265 │    100 │   2.65 │         11.0 │
└───────────────┴────────┴────────┴────────┴──────────────┘